# 16.1 Inverted indexes & Boolean retrieval

Search begins by turning documents into postings lists, so a query can jump straight to the documents that contain its terms.

This lesson is the first machinery behind retrieval: it teaches the data structure that makes keyword search fast. Sets and logic become the operations of retrieval: AND is intersection, OR is union, and NOT is set difference. Tokenization turns raw text into the terms that can be placed in an index. Save a copy to Drive to edit.

In [ ]:

import math
import re
import signal
import random
from collections import Counter
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

SEED = 16
random.seed(SEED)
np.random.seed(SEED)


def tokenize(text, lowercase=True, synonym_map=None, keep_case=False):
    if lowercase and not keep_case:
        text = text.lower()
    tokens = re.findall(r"[A-Za-z0-9_'-]+", text)
    if synonym_map is None:
        return tokens
    mapped = []
    for token in tokens:
        key = token.lower()
        mapped.append(synonym_map.get(key, key))
    return mapped


def ranked_doc_ids(scores, reverse=True):
    return [doc_id for doc_id, score in sorted(scores.items(), key=lambda item: (-item[1], item[0]) if reverse else (item[1], item[0]))]


def recall_at_k(ranking, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    hits = len(set(ranking[:k]) & relevant)
    return hits / len(relevant)


def dcg_at_k(ranking, gains, k):
    total = 0.0
    for rank, doc_id in enumerate(ranking[:k], start=1):
        gain = gains.get(doc_id, 0.0)
        total += gain / math.log2(rank + 1)
    return total


def ndcg_at_k(ranking, gains, k):
    ideal = sorted(gains.values(), reverse=True)
    ideal_total = 0.0
    for rank, gain in enumerate(ideal[:k], start=1):
        ideal_total += gain / math.log2(rank + 1)
    if ideal_total == 0:
        return 0.0
    return dcg_at_k(ranking, gains, k) / ideal_total


def mrr(ranking, relevant):
    relevant = set(relevant)
    for rank, doc_id in enumerate(ranking, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def preview_rungs(rungs):
    for rung in rungs:
        docs = rung["docs"]
        print(rung["name"], "docs=", len(docs), "query=", rung["query"])
        sample_ids = list(docs)[:3]
        for doc_id in sample_ids:
            print(" ", doc_id, "->", docs[doc_id][:90])
        print()


def try_fetch_20newsgroups_subset(categories, max_docs=18, timeout_seconds=3):
    def timeout_handler(signum, frame):
        raise TimeoutError("fetch_20newsgroups timed out")
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)
    try:
        from sklearn.datasets import fetch_20newsgroups
        data = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
        texts = [text.replace("\n", " ") for text in data.data[:max_docs]]
        return texts
    except Exception:
        return None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


def inline_newsgroups_docs():
    return {
        1: "space shuttle orbit nasa launch telescope mission",
        2: "hockey puck goalie playoff ice team",
        3: "graphics rendering image pixel shader gpu",
        4: "nasa mars orbit satellite telemetry",
        5: "goalie blocks puck in hockey playoff",
        6: "image compression rendering color pixel",
        7: "space telescope observes galaxy orbit",
        8: "team wins ice hockey tournament",
        9: "gpu shader renders three dimensional graphics",
        10: "satellite launch vehicle reaches orbit",
        11: "playoff team trades veteran goalie",
        12: "graphics image pipeline uses antialiasing",
    }


def lesson_docs_boolean():
    return {
        1: "neural search search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search search",
    }


def lesson_docs_sets():
    return {
        1: "neural search",
        2: "search graph",
        3: "neural retrieval",
        4: "graph retrieval search",
    }


def retrieval_ladder(kind):
    if kind == "boolean":
        return [
            {
                "name": "D1 lesson toy corpus",
                "docs": lesson_docs_sets(),
                "query": "search AND graph",
                "relevant": {2, 4},
            },
            {
                "name": "D2 12 clean topic docs",
                "docs": {
                    1: "neural search ranking",
                    2: "graph retrieval index",
                    3: "recipe ingredient oven",
                    4: "neural embeddings search",
                    5: "legal discovery contract",
                    6: "graph database traversal",
                    7: "search engine crawler",
                    8: "retrieval evaluation recall",
                    9: "support ticket reset",
                    10: "neural network training",
                    11: "index postings compression",
                    12: "semantic search retrieval",
                },
                "query": "neural AND search",
                "relevant": {1, 4},
            },
            {
                "name": "D3 case synonyms ties",
                "docs": {
                    1: "Search platform handles neural queries",
                    2: "semantic matching helps search",
                    3: "NEURAL retrieval benchmark",
                    4: "graph search retrieval",
                    5: "support SEARCH portal",
                    6: "neural Search ranking",
                    7: "lexical lookup only",
                    8: "semantic retrieval without keyword",
                    9: "search neural tie case",
                    10: "faq finder synonym lookup",
                    11: "audit log exclusion filter",
                    12: "rareterm alpha beta",
                },
                "query": "neural AND search",
                "relevant": {1, 6, 9},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space AND orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail negation rare terms",
                "docs": {
                    1: "urgent security incident xj9 raretoken search neural allowlist",
                    2: "security incident xj9 raretoken malware neural",
                    3: "Search incident raretoken benign customer support",
                    4: "security search xj9 raretoken not neural",
                    5: "compliance archive common logs",
                    6: "raretoken investigation search neural case study",
                    7: "neural search raretoken exclude deprecated",
                    8: "xj9 raretoken Search security search",
                    9: "incident response without keyword",
                    10: "rareterm unrelated graph retrieval",
                    11: "security neural search raretoken",
                    12: "deprecated neural search raretoken",
                },
                "query": "raretoken AND search AND NOT deprecated",
                "relevant": {1, 3, 4, 6, 8, 11},
            },
        ]
    if kind == "bm25":
        return [
            {
                "name": "D1 lesson BM25 corpus",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "gains": {1: 3, 3: 2, 4: 1, 2: 0},
            },
            {
                "name": "D2 clean keyword docs",
                "docs": {
                    1: "neural search ranking relevance",
                    2: "search ranking index",
                    3: "neural retrieval embeddings",
                    4: "graph database traversal",
                    5: "support search help center",
                    6: "neural network optimizer",
                    7: "enterprise search neural",
                    8: "retrieval evaluation ndcg",
                    9: "legal discovery query",
                    10: "search logs observability",
                    11: "neural search relevance",
                    12: "lexical index postings",
                },
                "query": "neural search",
                "gains": {1: 3, 7: 3, 11: 3, 3: 1, 6: 1, 2: 1, 5: 1, 10: 1},
            },
            {
                "name": "D3 stuffed repeats length variation",
                "docs": {
                    1: "neural search relevance",
                    2: "search search search search search coupon",
                    3: "neural retrieval scientific paper",
                    4: "search neural search neural ranking",
                    5: "very long document with search filler filler filler filler filler",
                    6: "graph traversal retrieval",
                    7: "neural semantic search",
                    8: "search support support support",
                    9: "neural neural model only",
                    10: "ranking index postings",
                    11: "search neural anti stuffing",
                    12: "irrelevant repeated repeated repeated",
                },
                "query": "neural search",
                "gains": {1: 3, 4: 3, 7: 3, 11: 3, 3: 1, 9: 1, 2: 0, 5: 0},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "gains": {1: 3, 4: 3, 7: 3, 10: 3},
            },
            {
                "name": "D5 long docs synonym misses",
                "docs": {
                    1: "neural search ranking tutorial with exact query terms",
                    2: "semantic retrieval finds meaning with embeddings and vectors",
                    3: "enterprise findability platform with semantic matching but no keyword overlap",
                    4: "neural neural neural search search stuffed landing page",
                    5: "long audit report search appendix appendix appendix appendix appendix appendix",
                    6: "support answer retrieval help article",
                    7: "semantic vector matching for neural information need",
                    8: "search logs with unrelated navigation data",
                    9: "dense encoder synonym recall for faq finder",
                    10: "legal discovery keyword exact match",
                    11: "neural search production benchmark",
                    12: "meaning based lookup with embeddings",
                },
                "query": "neural search",
                "gains": {1: 3, 2: 2, 3: 2, 7: 2, 9: 2, 11: 3, 4: 1},
            },
        ]
    if kind == "tfidf":
        return [
            {
                "name": "D1 lesson sparse vectors",
                "docs": lesson_docs_boolean(),
                "query": "neural search",
                "relevant": {1, 3, 4},
            },
            {
                "name": "D2 clean TF-IDF mini corpus",
                "docs": {
                    1: "neural search ranking",
                    2: "search index postings",
                    3: "neural embeddings retrieval",
                    4: "graph database",
                    5: "semantic search retrieval",
                    6: "support center",
                    7: "neural search evaluation",
                    8: "legal discovery",
                    9: "query expansion synonym",
                    10: "search logs",
                    11: "neural model",
                    12: "ranking relevance search",
                },
                "query": "neural search",
                "relevant": {1, 7, 3, 5, 11},
            },
            {
                "name": "D3 stopwords synonyms ties",
                "docs": {
                    1: "the neural search system",
                    2: "the car repair guide",
                    3: "automobile maintenance manual",
                    4: "search and the retrieval",
                    5: "neural retrieval and search",
                    6: "vehicle diagnostics handbook",
                    7: "the the the search",
                    8: "semantic lookup finder",
                    9: "neural model guide",
                    10: "the and of support",
                    11: "search neural duplicate",
                    12: "graph retrieval index",
                },
                "query": "neural search",
                "relevant": {1, 5, 9, 11},
            },
            {
                "name": "D4 20newsgroups-style 3 categories",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 long-tail vocab stopword-heavy docs",
                "docs": {
                    1: "car insurance claim policy vehicle accident",
                    2: "automobile coverage deductible crash report",
                    3: "the the the and of to in car",
                    4: "vehicle repair parts estimate",
                    5: "neural search unrelated tutorial",
                    6: "automobile automobile warranty service",
                    7: "car rental booking airport",
                    8: "rare sku xj9 part compatibility vehicle",
                    9: "support article about login reset",
                    10: "auto policy renewal notice",
                    11: "vehicle crash legal claim",
                    12: "long stopword document the and of in to with for on by",
                },
                "query": "car accident claim",
                "relevant": {1, 2, 4, 8, 10, 11},
            },
        ]
    if kind == "dense":
        return [
            {
                "name": "D1 lesson 2D embeddings",
                "docs": {
                    1: "semantic search match",
                    2: "faq synonym neighbor",
                    3: "orthogonal mismatch",
                    4: "opposite constraint",
                },
                "query": "semantic search",
                "embeddings": np.array([[1.0, 0.0], [0.8, 0.2], [0.0, 1.0], [-0.2, 0.9]]),
                "query_vector": np.array([0.9, 0.1]),
                "relevant": {1, 2},
            },
            {
                "name": "D2 clean semantic clusters",
                "docs": {i + 1: text for i, text in enumerate([
                    "password reset login help",
                    "account sign in recovery",
                    "invoice billing payment",
                    "credit card receipt",
                    "model training embeddings",
                    "vector retrieval semantic",
                    "shipping delivery package",
                    "order tracking courier",
                    "legal contract review",
                    "compliance policy audit",
                    "search ranking relevance",
                    "faq answer support",
                ])},
                "query": "login recovery",
                "relevant": {1, 2, 12},
            },
            {
                "name": "D3 synonyms noisy high-norm vectors",
                "docs": {i + 1: text for i, text in enumerate([
                    "car repair manual",
                    "automobile service guide",
                    "vehicle maintenance tips",
                    "billing payment invoice",
                    "login account help",
                    "huge norm noisy unrelated",
                    "auto insurance claim",
                    "engine diagnostic checklist",
                    "travel hotel booking",
                    "graph neural search",
                    "support ticket reset",
                    "semantic finder retrieval",
                ])},
                "query": "car maintenance",
                "relevant": {1, 2, 3, 7, 8},
            },
            {
                "name": "D4 20newsgroups tiny hashed SVD embeddings",
                "docs": inline_newsgroups_docs(),
                "query": "space orbit",
                "relevant": {1, 4, 7, 10},
            },
            {
                "name": "D5 off-domain constraint-heavy queries",
                "docs": {i + 1: text for i, text in enumerate([
                    "refund policy for enterprise plan excluding trials",
                    "trial cancellation guide no refund guaranteed",
                    "enterprise contract renewal refund clause",
                    "sports hockey playoff recap",
                    "graphics gpu shader discussion",
                    "refund exception for medical emergency",
                    "login reset support article",
                    "enterprise billing dispute with legal constraint",
                    "semantic neighbor about repayment but missing enterprise",
                    "off domain recipe ingredients",
                    "legal contract refund negotiation",
                    "high norm generic vector placeholder",
                ])},
                "query": "enterprise refund not trial",
                "relevant": {1, 3, 8, 11},
            },
        ]
    raise ValueError(kind)


def build_index(docs, lowercase=True, synonym_map=None):
    postings = defaultdict(set)
    tokenized = {}
    for doc_id, text in docs.items():
        tokens = tokenize(text, lowercase=lowercase, synonym_map=synonym_map)
        tokenized[doc_id] = tokens
        for token in sorted(set(tokens)):
            postings[token].add(doc_id)
    return {term: set(doc_ids) for term, doc_ids in postings.items()}, tokenized


def boolean_query(query, postings, universe, lowercase=True):
    parts = query.split()
    result = None
    op = "AND"
    negate_next = False
    for raw_part in parts:
        upper = raw_part.upper()
        if upper in {"AND", "OR"}:
            op = upper
            continue
        if upper == "NOT":
            negate_next = True
            continue
        term = raw_part.lower() if lowercase else raw_part
        current = set(postings.get(term, set()))
        if negate_next:
            current = set(universe) - current
            negate_next = False
        if result is None:
            result = current
        elif op == "AND":
            result = result & current
        else:
            result = result | current
    if result is None:
        return set()
    return result


## The concept, built once (D1)

An inverted index maps each term to a postings list, $P(t)=\{i:t\in d_i\}$. On the lesson corpus with $N=4$, Boolean retrieval is set arithmetic: $P(a\;	ext{AND}\;b)=P(a)\cap P(b)$, $P(a\;	ext{OR}\;b)=P(a)\cup P(b)$, and $P(	ext{NOT }a)=\{1,\ldots,N\}\setminus P(a)$.

In [ ]:

docs = lesson_docs_sets()
postings, tokenized = build_index(docs)
universe = set(docs)

assert len(docs) == 4
assert postings["neural"] == {1, 3}
assert postings["search"] == {1, 2, 4}
assert postings["graph"] == {2, 4}
assert postings["retrieval"] == {3, 4}

print("P(neural)=", postings["neural"])
print("P(search)=", postings["search"])
print("P(graph)=", postings["graph"])
print("P(retrieval)=", postings["retrieval"])


Now run the operators directly. The lesson examples are $\{1,2,4\}\cap\{2,4\}=\{2,4\}$ for `search AND graph`, $\{1,3\}\cup\{3,4\}=\{1,3,4\}$ for `neural OR retrieval`, and $\{1,2,3,4\}\setminus\{1,2,4\}=\{3\}$ for `NOT search`.

In [ ]:

search_and_graph = boolean_query("search AND graph", postings, universe)
neural_or_retrieval = boolean_query("neural OR retrieval", postings, universe)
not_search = boolean_query("NOT search", postings, universe)
plan_neural_and_search = boolean_query("neural AND search", postings, universe)
plan_graph_or_retrieval = boolean_query("graph OR retrieval", postings, universe)
plan_not_neural = boolean_query("NOT neural", postings, universe)

assert search_and_graph == {2, 4}
assert neural_or_retrieval == {1, 3, 4}
assert not_search == {3}
assert plan_neural_and_search == {1}
assert plan_graph_or_retrieval == {2, 3, 4}
assert plan_not_neural == {2, 4}

print("search AND graph:", search_and_graph)
print("neural OR retrieval:", neural_or_retrieval)
print("NOT search:", not_search)


## The dataset ladder

Family F14 has no shared ladder here, so this notebook builds D1-D5 inline: the lesson corpus, clean topic docs, case and synonym stress, a 20newsgroups-style three-category fallback corpus, and long-tail Boolean queries with negation and rare terms.

In [ ]:

rungs = retrieval_ladder("boolean")
preview_rungs(rungs)


In [ ]:

rows = []
artifacts = []
for rung in rungs:
    postings, tokenized = build_index(rung["docs"])
    universe = set(rung["docs"])
    result = boolean_query(rung["query"], postings, universe)
    ranking = sorted(result)
    score = recall_at_k(ranking, rung["relevant"], len(rung["docs"]))
    rows.append((rung["name"], score, ranking))
    artifacts.append((rung, postings, ranking))

print("rung | recall@k | matches")
for name, score, ranking in rows:
    print(f"{name} | {score:.3f} | {ranking}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, (rung, postings, ranking) in enumerate(artifacts):
    terms = sorted(postings)[:8]
    sizes = [len(postings[term]) for term in terms]
    axes[0, col].barh(terms, sizes)
    axes[0, col].set_title(rung["name"].split()[0])
    axes[0, col].set_xlabel("postings size")
    axes[1, col].bar(range(len(ranking)), [1] * len(ranking))
    axes[1, col].set_xticks(range(len(ranking)))
    axes[1, col].set_xticklabels(ranking)
    axes[1, col].set_ylim(0, 1.2)
    axes[1, col].set_title("Boolean matches")

metric_values = [row[1] for row in rows]
fig2, ax = plt.subplots(figsize=(6, 3))
ax.plot(["D1", "D2", "D3", "D4", "D5"], metric_values, marker="o")
ax.set_ylim(0, 1.05)
ax.set_ylabel("recall@k")
ax.set_title("Recall vs complexity")
plt.show()


## Pitfall on D5: forgetting normalization

If `Search` and `search` become separate keys, the D5 rare-term query silently loses candidates. Lowercasing and token normalization reunify the evidence.

In [ ]:

d5 = rungs[-1]
raw_postings, raw_tokens = build_index(d5["docs"], lowercase=False)
raw_result = boolean_query(d5["query"], raw_postings, set(d5["docs"]), lowercase=False)
raw_recall = recall_at_k(sorted(raw_result), d5["relevant"], len(d5["docs"]))
fixed_postings, fixed_tokens = build_index(d5["docs"], lowercase=True)
fixed_result = boolean_query(d5["query"], fixed_postings, set(d5["docs"]), lowercase=True)
fixed_recall = recall_at_k(sorted(fixed_result), d5["relevant"], len(d5["docs"]))

assert fixed_recall >= raw_recall
print("case-sensitive result", sorted(raw_result), "recall", raw_recall)
print("normalized result", sorted(fixed_result), "recall", fixed_recall)


## Evaluate it + Practice

- Report the planned metric (recall@k) beside a no-skill baseline such as random ranking or first-k document order.
- Run a cheap sanity check: the D1 asserts should pass before trusting any harder rung.
- Ablate the key idea and expect the metric to drop: turn off normalization or remove NOT handling and recall should fall on D5.
- Watch failure signals: empty candidates, tied scores, case splits, synonym misses, and high-norm artifacts.
- Keep all examples CPU-only, seeded, and small enough for a notebook cell.

Practice prompts:
1. Change one relevance label on D3 and recompute the metric table.



2. Add a new term to D2 and inspect how its postings list changes.

3. Replace one AND with OR on D5 and compare candidate-set size.